# IBBI PDF-to-text pipeline

This notebook converts the selected IBBI PDF into a clean, page-preserving text file. It also writes a JSON sidecar containing page-level text and extraction metadata for later auditing.

**Input PDF:** `24-aug-2026-2-d658bd6db947.pdf`

**Primary extractor:** PyMuPDF (`fitz`)

**Fallback:** optional OCR for pages that contain little or no extractable text.

## 1. Install dependencies

Install PyMuPDF in the same local virtual environment used by Jupyter. OCR dependencies are optional and should be installed only if the PDF contains scanned pages.

In [1]:
# Run this once in a terminal from the project virtual environment:
!python -m pip install pymupdf

# Optional OCR support, installed from the terminal if needed:
!python -m pip install pillow pytesseract
# Install the Tesseract executable separately and add it to PATH.

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.8 MB 1.3 MB/s eta 0:00:15
   -- ------------------------------------- 1.0/19.8 MB 1.6 MB/s eta 0:00:12
   -- ------------------------------------- 1.3/19.8 MB 1.6 MB/s eta 0:00:12
   --- ------------------------------------ 1.6/19.8 MB 1.6 MB/s eta 0:00:12
   ---- ----------------------------------- 2.4/19.8 MB 1.9 MB/s eta 0:00:10
   ----- ---------------------------------- 2.9/19.8 MB 2.0 MB/s eta 0:00:09
   ------- -------------------------------- 3.7/19.8 MB 2.2 MB/s eta 0:00:08
   -------- ------------------------------- 4.5/19.8 MB 2.4 MB/s eta 0:00:07
   ---------- ----------------------------- 5.2/19.8 MB 2.6 MB/s eta 0:00:06
   ------------ --------------------------- 6.3/19.8 MB 2.8 MB/s eta 0:00:05
   -------------- --

## 2. Select the PDF

This notebook is intended to run locally. The PDF is already stored in the experimental_setup folder.

In [2]:
from pathlib import Path

PDF_NAME = "24-aug-2026-2-d658bd6db947.pdf"
current_dir = Path.cwd()
if (current_dir / PDF_NAME).exists():
    WORK_DIR = current_dir
elif (current_dir / "experimental_setup" / PDF_NAME).exists():
    WORK_DIR = current_dir / "experimental_setup"
else:
    raise FileNotFoundError(
        "Run this notebook from the legal project root or experimental_setup folder."
    )

PDF_PATH = WORK_DIR / PDF_NAME
OUTPUT_DIR = WORK_DIR / "ibbi_text_output"

print(f"Input: {PDF_PATH}")
print(f"Exists: {PDF_PATH.exists()}")
if not PDF_PATH.exists():
    raise FileNotFoundError("The IBBI PDF was not found in the legal project or experimental_setup folder.")

Input: e:\Docs\intern\AAintern\legal\experimental_setup\24-aug-2026-2-d658bd6db947.pdf
Exists: True


## 3. Extraction functions

The cleaning step normalizes Unicode and spacing while retaining line and paragraph boundaries. Each page is kept separately so later experiments can trace extracted text back to the source page.

In [3]:
import json
import re
import unicodedata

import fitz


def clean_page_text(text: str) -> str:
    """Normalize extracted text without removing meaningful legal content."""
    text = unicodedata.normalize("NFKC", text or "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00ad", "")  # soft hyphen
    lines = []
    for line in text.split("\n"):
        line = re.sub(r"[ \t]+", " ", line).strip()
        lines.append(line)
    # Keep paragraph separation but remove excessive blank lines.
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def ocr_page(page) -> str:
    """OCR one page; requires optional pytesseract and Pillow packages."""
    import pytesseract
    from PIL import Image

    pixmap = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
    image = Image.frombytes("RGB", [pixmap.width, pixmap.height], pixmap.samples)
    return pytesseract.image_to_string(image)


def extract_pdf(pdf_path, output_dir, use_ocr=False, min_text_chars=40):
    """Extract a PDF and save page-preserving TXT and JSON outputs."""
    pdf_path = Path(pdf_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pages = []
    with fitz.open(pdf_path) as document:
        for page_number, page in enumerate(document, start=1):
            text = clean_page_text(page.get_text("text"))
            extraction_method = "text"
            ocr_error = None

            if use_ocr and len(text) < min_text_chars:
                try:
                    ocr_text = clean_page_text(ocr_page(page))
                    if len(ocr_text) > len(text):
                        text = ocr_text
                        extraction_method = "ocr"
                except Exception as exc:
                    ocr_error = f"{type(exc).__name__}: {exc}"

            page_record = {
                "page": page_number,
                "text": text,
                "characters": len(text),
                "extraction_method": extraction_method,
            }
            if ocr_error:
                page_record["ocr_error"] = ocr_error
            pages.append(page_record)

    text_with_page_markers = "\n\n".join(
        f"[PAGE {page['page']}]\n{page['text']}" for page in pages
    )
    result = {
        "source_file": pdf_path.name,
        "source_path": str(pdf_path),
        "page_count": len(pages),
        "total_characters": len(text_with_page_markers),
        "pages": pages,
        "text": text_with_page_markers,
    }

    txt_path = output_dir / f"{pdf_path.stem}.txt"
    json_path = output_dir / f"{pdf_path.stem}.json"
    txt_path.write_text(text_with_page_markers, encoding="utf-8")
    json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result, txt_path, json_path

## 4. Run the extraction

`USE_OCR_FALLBACK` is disabled initially because most IBBI orders are digitally generated PDFs. Enable it only when the page character counts show that some pages are empty or nearly empty.

In [4]:
USE_OCR_FALLBACK = False

result, text_path, json_path = extract_pdf(
    PDF_PATH,
    OUTPUT_DIR,
    use_ocr=USE_OCR_FALLBACK,
    min_text_chars=40,
)

print(f"Pages extracted: {result['page_count']}")
print(f"Total characters: {result['total_characters']}")
print(f"TXT saved to: {text_path}")
print(f"JSON saved to: {json_path}")

Pages extracted: 31
Total characters: 39527
TXT saved to: e:\Docs\intern\AAintern\legal\experimental_setup\ibbi_text_output\24-aug-2026-2-d658bd6db947.txt
JSON saved to: e:\Docs\intern\AAintern\legal\experimental_setup\ibbi_text_output\24-aug-2026-2-d658bd6db947.json


## 5. Inspect the extraction

This summary helps identify pages that may require OCR or manual review.

In [5]:
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

for page in result["pages"]:
    print(
        f"Page {page['page']:>3}: "
        f"{page['characters']:>6} characters, "
        f"method={page['extraction_method']}"
    )

print("\n--- First 2,000 extracted characters ---\n")
print(result["text"][:2000])

Page   1:    923 characters, method=text
Page   2:   1109 characters, method=text
Page   3:   1211 characters, method=text
Page   4:   1206 characters, method=text
Page   5:   1276 characters, method=text
Page   6:   1273 characters, method=text
Page   7:   1218 characters, method=text
Page   8:   1137 characters, method=text
Page   9:   1245 characters, method=text
Page  10:   1348 characters, method=text
Page  11:   1238 characters, method=text
Page  12:   1214 characters, method=text
Page  13:   2064 characters, method=text
Page  14:   1247 characters, method=text
Page  15:   1214 characters, method=text
Page  16:   1215 characters, method=text
Page  17:   1403 characters, method=text
Page  18:   1278 characters, method=text
Page  19:   1261 characters, method=text
Page  20:   1240 characters, method=text
Page  21:   1310 characters, method=text
Page  22:   1509 characters, method=text
Page  23:   1281 characters, method=text
Page  24:   1326 characters, method=text
Page  25:   1348